# 04 — Cost Model & Scenario Simulator

Quantifies analyst fatigue from false-positive triage burden
and simulates the effect of alert-volume reductions.

**All figures are MODELLED ESTIMATES** — see `docs/assumptions.md`.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.config import load_config, get_cost_params
from src.cost_model import (
    compute_cost_estimate,
    format_cost_report,
    compute_rule_cost_breakdown,
    simulate_reduction,
)

cfg = load_config()
db_path = cfg['paths']['database']
cost_params = get_cost_params(cfg)

In [ ]:
# Overall cost estimate
estimate = compute_cost_estimate(db_path, cost_params)
print(format_cost_report(estimate))

In [ ]:
# Per-rule cost breakdown
import pandas as pd
breakdown = compute_rule_cost_breakdown(db_path, cost_params)
df_cost = pd.DataFrame(breakdown)
df_cost.head(10)

In [ ]:
# Scenario simulation for top 3 FP-generating rules
top_rules = df_cost.nlargest(3, 'fp_count')['detection_rule'].tolist()

for rule in top_rules:
    print(f'\n=== {rule} ===')
    scenarios = simulate_reduction(db_path, detection_rule=rule, cost_params=cost_params)
    for s in scenarios:
        print(f'  Reduce by {s.reduction_pct}%: '
              f'save ~{s.hours_saved:.1f} hrs (${s.cost_saved:,.2f}), '
              f'~{s.estimated_tp_affected} TP affected')

In [ ]:
# Sensitivity analysis: vary triage time assumption
print('\n=== Sensitivity: varying triage minutes/alert ===')
for minutes in [10, 15, 20, 25, 30]:
    params = cost_params.copy()
    params['average_triage_minutes_per_alert'] = minutes
    est = compute_cost_estimate(db_path, params)
    annual = est.annualized_fp_hours or est.total_fp_triage_hours
    cost = est.annualized_labor_cost or est.estimated_labor_cost
    label = 'annual' if est.annualized_fp_hours else 'total'
    print(f'  {minutes} min/alert → {annual:,.0f} {label} hrs, ${cost:,.0f}')